# Run RWE on real MIND data (RQ2 / RQ3)

Downloads **MIND-small** from Microsoft's official source (research use — the data is *not* redistributed), ingests it, learns ideological positions from click behaviour, and runs the baselines + RWE-D/RWE-B to print the paper's **RQ2** (accuracy + long-tail) and **RQ3** (ideological-diversity) tables.

Runtime: a few minutes on a free CPU runtime. Nothing is committed — only the printed metrics / `results.csv`.

> If the GitHub repo is **private**, edit the clone URL in the next cell to include a token: `https://<TOKEN>@github.com/greenwichg/random_walks_with_erasure.git`

In [ ]:
# 1) Get the code (branch with the MIND pipeline) and install it
!git clone --branch claude/sleepy-gates-oecof1 https://github.com/greenwichg/random_walks_with_erasure.git
%cd random_walks_with_erasure
!pip install -e . -q
print('installed')

In [ ]:
# 1b) Drive cache — make every expensive artifact (the MIND data, lean.csv, the
#     .npz files) survive Colab runtime resets. Mounts Drive once; later cells
#     call cache_get / cache_put, so after the first successful run you never
#     re-download the data or re-run the GPU classifier again.
import os, shutil

CACHE = "/content/drive/MyDrive/rwe_mind"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CACHE, exist_ok=True)
    CACHE_OK = True
    print("Drive cache ready ->", CACHE)
except Exception as e:
    CACHE_OK = False
    print("(no Drive cache; artifacts will NOT persist across resets):", e)

def cache_get(name):
    """Copy <name> back from the Drive cache into the working dir if available."""
    src = os.path.join(CACHE, os.path.basename(name))
    if CACHE_OK and os.path.exists(src) and not os.path.exists(name):
        shutil.copy(src, name)
        print("restored from Drive cache:", name)
    return os.path.exists(name)

def cache_put(name):
    """Save <name> to the Drive cache for future runs."""
    if CACHE_OK and os.path.exists(name):
        shutil.copy(name, os.path.join(CACHE, os.path.basename(name)))
        print("cached to Drive:", name)

In [ ]:
# 2) Get MIND-small (train). Microsoft GATED the public blob (HTTP 409: "Public
#    access is not permitted"), so the old direct download no longer works for
#    anyone. This cell reuses the Drive cache -> tries the official URL -> falls
#    back to an inline upload, then caches the zip to Drive.
#
#    If it is not already cached, get MINDsmall_train.zip ONCE from a mirror:
#      * Kaggle  - search "MIND microsoft news" (free login), download the train zip
#      * Hugging Face - huggingface.co/datasets, search "MIND"
#    then pick it in the upload dialog this cell pops up.
import os, glob, urllib.request
ZIP = "MINDsmall_train.zip"

def have_news():
    return [h for h in glob.glob("**/news.tsv", recursive=True) if "fixture" not in h]

if not have_news():
    if not cache_get(ZIP):                       # not in cwd and not in Drive cache
        try:
            url = "https://mind201910small.blob.core.windows.net/release/" + ZIP
            print("trying official source (often gated now) ...")
            urllib.request.urlretrieve(url, ZIP)
        except Exception as e:
            print("official source unavailable:", e)
            print("Pick the MINDsmall_train.zip you downloaded from a mirror:")
            from google.colab import files
            up = files.upload()                  # inline file picker
            zips = [f for f in up if f.lower().endswith(".zip")]
            if zips and zips[0] != ZIP:
                os.replace(zips[0], ZIP)
        cache_put(ZIP)                           # persist for future runs
    os.system("unzip -q -o %s -d MINDsmall_train" % ZIP)

assert have_news(), "still no news.tsv - provide MINDsmall_train.zip per the notes above"
print("MIND ready ->", have_news()[0])

In [ ]:
# 3) Ingest: click graph + political tagging + co-click ideological positions
#    (the ideal-point fit is the slow step). Cached to Drive -> skipped on reruns.
#    NOTE: this co-click axis tends to be TOPICAL; the text-lean path (cells 7-8)
#    is the recommended one for the headline numbers.
import glob, os
if not cache_get("mind.npz"):
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    print('using MIND_DIR =', MIND_DIR)
    get_ipython().system(f"python examples/ingest_mind.py --mind-dir {MIND_DIR} --political-only --ideology --min-user-clicks 10 --min-item-clicks 10 --sample-users 15000 --out mind.npz")
    cache_put("mind.npz")
# Watch the printed lean_corr: closer to 1.0 = the latent axis is left-right.

> 📐 **Look at the `AXIS ALIGNMENT` block** in the output below — a `Pearson r` near **+1** and a high **expected-side %** confirm left-leaning users sit on the left (the axis isn't sign-flipped). Copy that number into `docs/RESULTS.md`.

In [ ]:
# 4) Evaluate: baselines + RWE-D/RWE-B -> RQ2 & RQ3 tables + results.csv
#    (remove --no-bprmf to add the slower BPRMF baseline)
!python examples/eval_mind.py --npz mind.npz --out-csv results.csv --no-bprmf

In [ ]:
# 5) Show the full results table and download the CSV
import pandas as pd
df = pd.read_csv('results.csv', index_col=0)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50)
print(df.round(3).to_string())
try:
    from google.colab import files; files.download('results.csv')
except Exception:
    pass

## Bounded-bridging sweep (the extension's key experiment)

Vary RWE-B's *not too far* bound `d` and watch whether bridging stays strong (`uw_shift` high) while recommendations move back toward the centre (`uw_recs` low) instead of the opposite extreme (the `d=inf` row). This is the real-data test of the bounded-bridging idea in `rwe/opinion_dynamics.py`.

In [ ]:
# 6) RWE-B bounded-bridging sweep (reuses mind.npz; no re-ingest)
!python examples/eval_mind.py --npz mind.npz --out-csv sweep.csv \
  --sweep-max-distance 3,2,1.5,1,0.5 --no-bprmf
import pandas as pd
print(pd.read_csv('sweep.csv', index_col=0).round(3).to_string())

## Option B — text-grounded ideology axis (recommended)

The co-click `--ideology` axis above turns out **topical**, not left-right (check the headline eyeball). This section scores each article's lean from its **text** (title + abstract) with a pretrained classifier, uses that as the ideological axis, and re-runs eval + the sweep — so RQ3 is about real lean.

**Switch to a GPU runtime first**: Runtime -> Change runtime type -> GPU.

In [ ]:
# 7) Score each political article's lean from its TEXT (GPU recommended:
#    Runtime -> Change runtime type -> GPU). Cached to Drive, so after a reset
#    this slow step is skipped automatically.
import glob, os
if not cache_get("lean.csv"):
    get_ipython().system("pip install -q transformers")
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    get_ipython().system(f"python examples/classify_lean.py --mind-dir {MIND_DIR} --political-only --out lean.csv")
    cache_put("lean.csv")
else:
    print("using cached lean.csv (skipped the GPU classifier)")
# eyeball the 'Most LEFT/RIGHT-scored' headlines it prints -- they should look ideological

In [ ]:
# 8) Re-position from the text lean (no --ideology), then eval + sweep. The .npz
#    is cached to Drive (reset-safe); the fast eval/sweep always run.
import glob, os
if not cache_get("mind_text.npz"):
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    get_ipython().system(f"python examples/ingest_mind.py --mind-dir {MIND_DIR} --political-only --positions-csv lean.csv --min-user-clicks 10 --min-item-clicks 10 --sample-users 15000 --out mind_text.npz")
    cache_put("mind_text.npz")
get_ipython().system("python examples/eval_mind.py --npz mind_text.npz --out-csv results_text.csv --no-bprmf")
get_ipython().system("python examples/eval_mind.py --npz mind_text.npz --out-csv sweep_text.csv --sweep-max-distance 3,2,1.5,1,0.5 --no-bprmf")
import pandas as pd
print('RESULTS (text-lean axis):')
print(pd.read_csv('results_text.csv', index_col=0).round(3).to_string())
print()
print('SWEEP (text-lean axis):')
print(pd.read_csv('sweep_text.csv', index_col=0).round(3).to_string())

In [ ]:
# 8b) Robustness: average over 7 seeds + Wilcoxon significance vs P3
#     (prints mean +/- std tables and p-values; writes *_std / *_pvalues)
!python examples/eval_mind.py --npz mind_text.npz --seeds 7 --no-bprmf \
  --out-csv results_text_ms.csv

In [ ]:
# 8c) Plot where users and items actually sit on the left<->right scale
#     (left-leaning on the left, right-leaning on the right). Uses the
#     text-lean mind_text.npz; saves + shows axis.png.
get_ipython().system("python examples/plot_axis.py --npz mind_text.npz --out axis.png")
from IPython.display import Image, display
display(Image("axis.png"))
try:
    from google.colab import files; files.download("axis.png")
except Exception:
    pass

In [ ]:
# 8e) (optional, GPU) Reporting-vs-opinion register -> register.csv, for the
#     report's Reporting Ratio. Zero-shot (bart-large-mnli); cached to Drive.
import glob, os
if not cache_get("register.csv"):
    get_ipython().system("pip install -q transformers")
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    get_ipython().system(f"python examples/classify_register.py --mind-dir {MIND_DIR} --political-only --out register.csv")
    cache_put("register.csv")

In [ ]:
# 8f) (optional, GPU, EXPERIMENTAL) Emotional tone -> emotion.csv, for the
#     report's Attention profile / Emotional Balance. Emotion-from-headline is
#     NOISY -- treat as low-confidence (see docs/HEALTH_REPORT_PLAN.md). Cached.
import glob, os
if not cache_get("emotion.csv"):
    get_ipython().system("pip install -q transformers")
    MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
    get_ipython().system(f"python examples/classify_emotion.py --mind-dir {MIND_DIR} --political-only --out emotion.csv")
    cache_put("emotion.csv")

In [ ]:
# 8d) Information Health Report — reading-diet profile from mind_text.npz.
#     register.csv / emotion.csv (cells 8e/8f) add Reporting / Emotional /
#     Attention; --behaviors adds Open-Mindedness (cross-cutting click-through).
#     PoC; see docs/HEALTH_REPORT.md.
import os, glob
extra = ""
if os.path.exists("register.csv"): extra += " --register-csv register.csv"
if os.path.exists("emotion.csv"):  extra += " --emotion-csv emotion.csv"
beh = [h for h in glob.glob("**/behaviors.tsv", recursive=True) if "fixture" not in h]
if beh: extra += f" --behaviors {beh[0]}"
get_ipython().system(f"python examples/health_report.py --npz mind_text.npz --sample 3 --html health_report.html{extra}")
from IPython.display import HTML, display
display(HTML(open("health_report.html").read()))
try:
    from google.colab import files; files.download("health_report.html")
except Exception:
    pass

## Option C — how ideological is the axis? (validate it)

The text-lean axis is a noisy proxy. Quantify it: sample a few articles, label them yourself (without peeking at the model), and correlate. Or score with a second bias model and correlate the two — see `examples/validate_lean.py`.

In [ ]:
# 9) Make a 40-article labeling template, download it, label offline
import glob, os
MIND_DIR = os.path.dirname([h for h in glob.glob('**/news.tsv', recursive=True) if 'fixture' not in h][0])
!python examples/validate_lean.py --lean lean.csv --news-dir {MIND_DIR} --sample 40 --out label_template.tsv
from google.colab import files; files.download('label_template.tsv')
# fill the 'position' column (-1/0/1), re-upload, then run:
#   !python examples/validate_lean.py --lean lean.csv --against label_template.tsv